In [2]:
import cv2
import numpy as np
import os

input_folder = "enhanced_frames"
output_folder = "output_edge_cracks"

os.makedirs(output_folder, exist_ok=True)

def road_edge_mask(gray):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)

    # focus ONLY on left and right road edges
    mask[int(h*0.45):h, :int(w*0.25)] = 255
    mask[int(h*0.45):h, int(w*0.75):] = 255

    return mask

def detect_edge_cracks(gray, output, mask):

    roi = cv2.bitwise_and(gray, gray, mask=mask)

    blur = cv2.GaussianBlur(roi, (5,5), 0)

    edges = cv2.Canny(blur, 40, 120)

    crack_map = np.zeros_like(edges)

    h, w = edges.shape

    block_w = 40
    block_h = 60

    for y in range(0, h - block_h, 20):
        for x in range(0, w - block_w, 20):

            region = edges[y:y+block_h, x:x+block_w]

            density = np.mean(region > 0)

            if density > 0.08:
                crack_map[y:y+block_h, x:x+block_w] = 255

    kernel = np.ones((15,15), np.uint8)

    crack_map = cv2.morphologyEx(crack_map, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(crack_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 2000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        aspect = h / (w + 1e-5)

        if aspect < 1.2:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (255,255,0), 2)

        cv2.putText(
            output,
            "Edge Crack",
            (x, y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,0),
            2
        )

        count += 1

    return count

files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png','.jpg','.jpeg')):
        continue

    path = os.path.join(input_folder, file)

    frame = cv2.imread(path)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    output = frame.copy()

    mask = road_edge_mask(gray)

    count = detect_edge_cracks(gray, output, mask)

    cv2.putText(
        output,
        f"Edge Cracks: {count}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,255),
        2
    )

    cv2.imwrite(os.path.join(output_folder, file), output)

    print(f"{file} → {count} edge cracks")

print("\n EDGE CRACK DETECTION COMPLETE")

AlligatorCracking_frame_0.jpg → 1 edge cracks
AlligatorCracking_frame_20.jpg → 1 edge cracks
AlligatorCracking_frame_35.jpg → 1 edge cracks
AlligatorCracking_frame_4.jpg → 1 edge cracks
AlligatorCracking_frame_51.jpg → 1 edge cracks
AlligatorCracking_frame_9.jpg → 1 edge cracks
EdgeCracking3_frame_72.jpg → 2 edge cracks
EdgeCracking4_frame_123.jpg → 2 edge cracks
EdgeCracking4_frame_129.jpg → 2 edge cracks
EdgeCracking4_frame_130.jpg → 2 edge cracks
EdgeCracking4_frame_133.jpg → 2 edge cracks
EdgeCracking4_frame_144.jpg → 2 edge cracks
EdgeCracking4_frame_154.jpg → 2 edge cracks
EdgeCracking9_frame_185.jpg → 2 edge cracks
EdgeCracking9_frame_202.jpg → 2 edge cracks
EdgeCracking9_frame_206.jpg → 2 edge cracks
EdgeCracking9_frame_217.jpg → 2 edge cracks
RoadPotholes2_frame_522.jpg → 2 edge cracks
RoadPotholes3_frame_575.jpg → 2 edge cracks
RoadPotholes3_frame_578.jpg → 2 edge cracks
RoadPotholes3_frame_585.jpg → 3 edge cracks
RoadPotholes_frame_478.jpg → 2 edge cracks
RoadPotholes_frame_